# Pointwise lower bounds for the number of shapes

Companion to Section IV-C of *Asymptotic Growth of the Number of Rubik's Snake Shapes*. This notebook is independently reproducible with the standard library, `../rubiks_snake.py`, NumPy, Numba, and SciPy, plus Jupyter. It does not load another notebook or any unpublished file.

We regenerate exact slab tables and irreducibles. `counts[ell]` counts selected irreducible blocks of total length $j=\ell+1$. Their concatenation sequence satisfies $c_0=1$, $c_k=\sum_\ell\text{counts}[\ell]c_{k-\ell-1}$, with $c_k=0$ for negative $k$. Unique factorization and the two endpoint wedges give **$c_{n-2}\leq S_n$**.

Let $L=\text{len(counts)}$ and choose a certified $q$ with $\sum_\ell \text{counts}[\ell]q^{-(\ell+1)}>1$. Checking $c_k\geq a q^k$ for `start <= k < start+L` proves the same for every $k\geq\text{start}$ by induction. The function computes the minimum of $c_k/q^k$ on that finite base using exact fractions, then rounds **down** to an easily reported rational. The pointwise exponential bound starts at `n=start+2`.

**Outputs and timing.** Small examples were reproduced on 2026-09-19 with the repository interpreter. The saved final output is an archived research reference from 2026-09-11; its exact coefficient and induction checks also pass the current regression tests, but the full notebook cell was not rerun. Execution counts are null. Rerunning recomputes every slab table and replaces that output. The current function times its own total, including first-use compilation. Counts are checked for integer overflow.

In [ ]:
from pathlib import Path
from time import perf_counter
from fractions import Fraction
import sys
started = perf_counter()
cwd = Path.cwd()
candidates = (cwd, cwd / 'rubiks-snake', cwd.parent, cwd.parent / 'rubiks-snake')
snake_dir = next((p for p in candidates if (p / 'rubiks_snake.py').is_file()), None)
if snake_dir is None:
    raise FileNotFoundError('Run from asymptotic-analysis, rubiks-snake, or the repository root')
sys.path.insert(0, str(snake_dir.resolve()))
import numpy as np
import numba
import scipy
from rubiks_snake import (slab_counts, irreducible_slab_counts, renewal_lower_bound,
                         renewal_lower_prefactor, renewal_values)
print(f'Python {sys.version.split()[0]}; NumPy {np.__version__}; Numba {numba.__version__}; SciPy {scipy.__version__}')
print(f'Setup: {perf_counter() - started:.3f} s')

In [ ]:
def get_bound(cutoffs, denominator=10**9, factor_denominator=250, start=2):
    """Return a, q proving S_n >= a*q**(n-2) for n>=start+2."""
    if not cutoffs or sorted(cutoffs) != list(range(1, len(cutoffs) + 1)):
        raise ValueError('Use consecutive progress values d=1,...,D')
    limits = [cutoffs[d] for d in sorted(cutoffs)]
    if any(not isinstance(k, int) or k < 1 for k in limits) or limits != sorted(limits, reverse=True):
        raise ValueError('Internal-length cutoffs must be positive and nonincreasing')
    if not isinstance(factor_denominator, int) or factor_denominator < 1:
        raise ValueError('factor_denominator must be a positive integer')
    started = perf_counter()
    raw = {d: slab_counts(d - 1, cutoffs[d]) for d in sorted(cutoffs)}
    counts = irreducible_slab_counts(raw)
    q = renewal_lower_bound(counts, denominator)
    exact_factor = renewal_lower_prefactor(counts, q, start)
    a = Fraction(exact_factor.numerator * factor_denominator // exact_factor.denominator, factor_denominator)
    if a <= 0:
        raise ValueError('Increase factor_denominator to retain a positive prefactor')
    base = renewal_values(counts, start + len(counts) - 1)
    assert all(Fraction(base[k]) >= a * q**k for k in range(start, len(base)))
    elapsed = perf_counter() - started
    print(f'Exact induction base checked: {start} <= k <= {len(base)-1}')
    print(f'S_n >= ({a}) * ({q})**(n-2) for n >= {start+2}; {elapsed:.3f} s')
    return dict(bound=q, factor=a, counts=counts, first_n=start+2, seconds=elapsed)

def integer_lower_bounds(result, lengths):
    """Evaluate the stronger renewal counts, not a floating-point exponential."""
    if not lengths or any(not isinstance(n, int) or n < 2 for n in lengths):
        raise ValueError('Supply wedge lengths n >= 2')
    started = perf_counter()
    values = renewal_values(result['counts'], max(lengths) - 2)
    for n in lengths:
        print(f'n={n}: {values[n-2]} <= S_n')
    print(f'Renewal evaluation: {perf_counter() - started:.3f} s')

In [ ]:
SMALL_CASES = [{1: 4}, {1: 8}, {1: 12}, {1: 16, 2: 12, 3: 10}]
small_results = [get_bound(cutoffs) for cutoffs in SMALL_CASES]
integer_lower_bounds(small_results[-1], [4, 10, 14, 28])

Exact induction base checked: 2 <= k <= 6
S_n >= (71/250) * (1518630369/500000000)**(n-2) for n >= 4; 0.000 s
Exact induction base checked: 2 <= k <= 10
S_n >= (32/125) * (783771809/250000000)**(n-2) for n >= 4; 0.000 s
Exact induction base checked: 2 <= k <= 14
S_n >= (32/125) * (3143484601/1000000000)**(n-2) for n >= 4; 0.000 s
Exact induction base checked: 2 <= k <= 18
S_n >= (51/250) * (1654667871/500000000)**(n-2) for n >= 4; 0.019 s
n=4: 4 <= S_n
n=10: 3024 <= S_n
n=14: 399568 <= S_n
n=28: 7436586747712 <= S_n
Renewal evaluation: 0.000 s


## Published bound and how to improve it

Increase `PUBLISHED_CUTOFFS` to strengthen the block family, preserving nonincreasing cutoffs. `factor_denominator` controls how much of the exact prefactor is retained. A better base can require a smaller prefactor; pointwise bounds at every short length need not improve when the base improves. The integer recurrence remains an independently useful lower bound.

In [ ]:
PUBLISHED_CUTOFFS = {1: 28, 2: 20, 3: 18}
published = get_bound(PUBLISHED_CUTOFFS, factor_denominator=250)
integer_lower_bounds(published, [4, 10, 14, 28])
q, a = published['bound'], published['factor']
if PUBLISHED_CUTOFFS == {1: 28, 2: 20, 3: 18}:
    assert q > Fraction(34000, 10000) and a / q**2 > Fraction(14, 1000)
    print('Rounded corollary verified exactly: S_n > 0.014*(3.4000)**n for n >= 4')

Archived reference result (2026-09-11):
Exact induction base checked: 2 <= k <= 30
S_n >= (41/250) * (3400034903/1000000000)**(n-2) for n >= 4
n=4: 4 <= S_n
n=10: 3024 <= S_n
n=14: 399568 <= S_n
n=28: 11951167017736 <= S_n
Rounded corollary verified exactly: S_n > 0.014*(3.4000)**n for n >= 4
